Sarah Sullivan

October 23, 2025

Last Updated: Jan 20, 2026

PSID 

In [1]:
# Import necessary packages
import pandas as pd
import numpy as np
import ast

In [2]:
# Define root
root = "/Users/sarsul/Library/CloudStorage/Dropbox-UniversityofMichigan/Sarah Sullivan/SARAH-SOFT/research/psid/"

In [3]:
df = pd.read_stata(root + "A1_try_pyth_v1.dta")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/sarsul/Library/CloudStorage/Dropbox-UniversityofMichigan/Sarah Sullivan/SARAH-SOFT/research/psid/A1_try_pyth_v1.dta'

In [4]:
# define prior year's household roster and ages of people in that household roster

df['hhr_prev'] = df['hhr_'].shift(1)
df['ages_prev'] = df['ages'].shift(1)

# set each person's first observation hhr and ages of hhr to empty (no observed past roster at time 0)

df.loc[df['ID'] != df['ID'].shift(1), 'hhr_prev'] = df.loc[df['ID'] != df['ID'].shift(1), 'hhr_prev'].apply(lambda x: [])
df.loc[df['ID'] != df['ID'].shift(1), 'ages_prev'] = df.loc[df['ID'] != df['ID'].shift(1), 'ages_prev'].apply(lambda x: [])


/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42296/758206761.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['hhr_prev'] = df['hhr_'].shift(1)
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42296/758206761.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ages_prev'] = df['ages'].shift(1)


In [5]:
# Fn to convert tuples to lists of strings

def parse_tuple_string(x):
    if isinstance(x, str):
        try:
            result = ast.literal_eval(x)
            if isinstance(result, (list, tuple)):
                return list(result)
            else:
                return [result]  
        except (SyntaxError, ValueError):
            return []
    elif isinstance(x, (list, tuple)):
        return list(x)
    else:
        return [x] if x is not None else []

In [6]:
# pass hhr and ages variables into above

df['hhr_prev'] = df['hhr_prev'].apply(parse_tuple_string)
df['hhr_'] = df['hhr_'].apply(parse_tuple_string)

df['ages_prev'] = df['ages_prev'].apply(parse_tuple_string) 
df['ages'] = df['ages'].apply(parse_tuple_string)


In [7]:
# subtract previous household roster from current to see who left
# subtract current household roster from previous to see who came

df = df.copy()

df['who_left'] = df.apply(lambda row: [] if not row['hhr_prev'] else [x for x in row['hhr_prev'] if x not in row['hhr_']], axis=1)

df['who_came'] = df.apply(lambda row: [] if not row['hhr_prev'] else [x for x in row['hhr_'] if x not in row['hhr_prev']], axis=1)

In [8]:
def get_ages_left(row):
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    ages_prev = row['ages_prev'] if isinstance(row['ages_prev'], list) else []
    who_left = row['who_left'] if isinstance(row['who_left'], list) else []
    return [ages_prev[hhr_prev.index(id)] for id in who_left if id in hhr_prev and hhr_prev.index(id) < len(ages_prev)]

def get_ages_came(row):
    hhr_ = row['hhr_'] if isinstance(row['hhr_'], list) else []
    ages = row['ages'] if isinstance(row['ages'], list) else []
    who_came = row['who_came'] if isinstance(row['who_came'], list) else []
    return [ages[hhr_.index(pid)] for pid in who_came if pid in hhr_ and hhr_.index(pid) < len(ages)]

In [9]:
df['ages_left'] = df.apply(get_ages_left, axis=1)
df['ages_came'] = df.apply(get_ages_came, axis=1)

In [10]:
df['adult_came'] = df['ages_came'].apply(
    lambda ages: int(isinstance(ages, list) and any(age >= 18 for age in ages))
)

df['child_came'] = df['ages_came'].apply(
    lambda ages: int(isinstance(ages, list) and any(age < 18 for age in ages))
)

df['adult_left'] = df['ages_left'].apply(
    lambda ages: int(isinstance(ages, list) and any(age >= 18 for age in ages))
)

df['child_left'] = df['ages_left'].apply(
    lambda ages: int(isinstance(ages, list) and any(age < 18 for age in ages))
)

In [11]:
df['n_adults_left'] = df['ages_left'].apply(
    lambda ages: sum(age >= 18 for age in ages) if isinstance(ages, list) else 0
)

df['n_adults_came'] = df['ages_came'].apply(
    lambda ages: sum(age >= 18 for age in ages) if isinstance(ages, list) else 0
)

In [12]:
sib_cols = [f'ID_S{str(i).zfill(2)}' for i in range(1, 17)]
df['sib_list'] = df[sib_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)

In [13]:
df['sib_came'] = df.apply(
    lambda row: int(isinstance(row['who_came'], list) and isinstance(row['sib_list'], list) and any(id in row['sib_list'] for id in row['who_came'])),
    axis=1
)

df['sib_left'] = df.apply(
    lambda row: int(isinstance(row['who_left'], list) and isinstance(row['sib_list'], list) and any(id in row['sib_list'] for id in row['who_left'])),
    axis=1
)   

In [14]:
# ages came
def get_sib_ages_came(row):
    if not isinstance(row['who_came'], list) or not isinstance(row['sib_list'], list):
        return []
    hhr_ = row['hhr_'] if isinstance(row['hhr_'], list) else []
    ages = row['ages'] if isinstance(row['ages'], list) else []
    # Get IDs that are both in who_came and sib_list
    sibs_who_came = [id for id in row['who_came'] if id in row['sib_list']]
    # Get ages for those siblings
    return [ages[hhr_.index(sib_id)] for sib_id in sibs_who_came if sib_id in hhr_ and hhr_.index(sib_id) < len(ages)]

# ages left
def get_sib_ages_left(row):
    if not isinstance(row['who_left'], list) or not isinstance(row['sib_list'], list):
        return []
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    ages_prev = row['ages_prev'] if isinstance(row['ages_prev'], list) else []
    # Get IDs that are both in who_left and sib_list
    sibs_who_left = [id for id in row['who_left'] if id in row['sib_list']]
    # Get ages for those siblings
    return [ages_prev[hhr_prev.index(sib_id)] for sib_id in sibs_who_left if sib_id in hhr_prev and hhr_prev.index(sib_id) < len(ages_prev)]

df['sib_ages_came'] = df.apply(get_sib_ages_came, axis=1)
df['sib_ages_left'] = df.apply(get_sib_ages_left, axis=1)

In [15]:
# Create gpar_list: list of ID's for all grandparents
gpar_cols = ['ID_aM_aM', 'ID_aM_aD', 'ID_aM_bM', 'ID_aM_bD', 'ID_aD_aM', 'ID_aD_aD', 'ID_aD_bM', 'ID_aD_bD', 
             'ID_bM_aM', 'ID_bM_aD', 'ID_bM_bM', 'ID_bM_bD', 'ID_bD_aM', 'ID_bD_aD', 'ID_bD_bM', 'ID_bD_bD']
df['gpar_list'] = df[gpar_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)

In [16]:
# Create par_list: list of ID's for parents
par_cols = ['ID_aM', 'ID_aD', 'ID_bM', 'ID_bD']
df['par_list'] = df[par_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)

In [17]:
df['gpar_came'] = df.apply(
    lambda row: int(isinstance(row['who_came'], list) and isinstance(row['gpar_list'], list) and any(id in row['gpar_list'] for id in row['who_came'])),
    axis=1
)

df['gpar_left'] = df.apply(
    lambda row: int(isinstance(row['who_left'], list) and isinstance(row['gpar_list'], list) and any(id in row['gpar_list'] for id in row['who_left'])),
    axis=1
)

In [18]:
df['par_came'] = df.apply(
    lambda row: int(isinstance(row['who_came'], list) and isinstance(row['par_list'], list) and any(id in row['par_list'] for id in row['who_came'])),
    axis=1
)

df['par_left'] = df.apply(
    lambda row: int(isinstance(row['who_left'], list) and isinstance(row['par_list'], list) and any(id in row['par_list'] for id in row['who_left'])),
    axis=1
)

In [19]:
# Create relatives list
rel_cols = sib_cols + par_cols + gpar_cols

In [20]:
df['rel_list'] = df[rel_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)

In [21]:
df['other_came'] = df.apply(
    lambda row: int(isinstance(row['who_came'], list) and isinstance(row['rel_list'], list) and any(id not in row['rel_list'] for id in row['who_came'])),
    axis=1
)

df['other_left'] = df.apply(
    lambda row: int(isinstance(row['who_left'], list) and isinstance(row['rel_list'], list) and any(id not in row['rel_list'] for id in row['who_left'])),
    axis=1
)

In [22]:
df.to_csv('A1_python_v1.csv', index=False)